In [ ]:
import pandas as pd
import numpy as np
import talib
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import bt
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


from statsmodels.tsa.stattools import acf, adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# Load Data

In [ ]:
# Load BCA data
tickers = ["BBCA.JK"]
bbca = yf.Ticker("BBCA.JK")
data = bbca.history(start="2020-01-01")
data

In [ ]:
# Split data into training and testing sets
train_data = data.iloc[:int(0.8 * len(data))]
test_data = data.iloc[int(0.8 * len(data)):]

# Trading Strategy

## Regime-Switching Strategy

Based on our EDA scorecard, BBCA exhibits **mixed behavior**:
- **Rolling Hurst**: 82.2% of the time in trending regime (H > 0.5)
- **Variance Ratio**: VR = 0.76 → mean-reverting at multiple horizons
- **Half-life**: ~13.6 days for SMA(50) spread → fast mean reversion

**Strategy Design:**

| Regime | Detector | Strategy | Logic |
|---|---|---|---|
| Trending (H > 0.5) | Rolling Hurst | **EMA Crossover + ADX** | Long when short EMA > long EMA AND trend is strong |
| Mean Reverting (H ≤ 0.5) | Rolling Hurst | **Bollinger Band Z-Score** | Long when price is oversold (Z < -1), exit at mean (Z > 0) |

### Component 1: Trend Following — Donchiat Breakout

In [ ]:
def donchian_breakout_signal(df, window=20):
    """
    Trend Following: Donchian Channel Breakout.
    Long when Close breaks highest high of past N days.
    Exit (go flat) when Close breaks lowest low of past N days.
    """
    high_roll = df['High'].rolling(window).max()
    low_roll = df['Low'].rolling(window).min()
    signal = np.zeros(len(df), dtype=int)

    # Donchian logic: Entry/Exit
    long_entries = df['Close'] > high_roll.shift(1)
    exits = df['Close'] < low_roll.shift(1)

    in_trade = False
    for i in range(len(df)):
        if not in_trade and long_entries.iloc[i]:
            in_trade = True
        elif in_trade and exits.iloc[i]:
            in_trade = False
        signal[i] = 1 if in_trade else 0

    donchian_high = high_roll
    donchian_low = low_roll
    return pd.Series(signal, index=df.index), donchian_high, donchian_low

tf_signal, donchian_high, donchian_low = donchian_breakout_signal(train_data, window=20)

# --- Visualize ---
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(train_data.index, train_data['Close'], color='gray', alpha=0.7, label='Close')
axes[0].plot(train_data.index, donchian_high, color='blue', lw=1, label='Donchian High (20)')
axes[0].plot(train_data.index, donchian_low, color='red', lw=1, label='Donchian Low (20)')
axes[0].fill_between(train_data.index, train_data['Close'].min(), train_data['Close'].max(),
                      where=tf_signal == 1, alpha=0.1, color='green', label='Long')
axes[0].set_title('Trend Following: Donchian Breakout (20-day)')
axes[0].set_ylabel('Price')
axes[0].legend(loc='upper left', fontsize=8)

axes[1].plot(train_data.index, donchian_high, color='blue', lw=1, label='Donchian High')
axes[1].plot(train_data.index, donchian_low, color='red', lw=1, label='Donchian Low')
axes[1].set_title('Donchian Channel Bounds')
axes[1].set_ylabel('Price')
axes[1].legend()

axes[2].plot(train_data.index, tf_signal, color='green', lw=0.8)
axes[2].set_title('Signal (1=Long, 0=Flat)')
axes[2].set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

# Quick perf
ret = np.log(train_data['Close']).diff().shift(-1)
tf_ret = (tf_signal * ret).dropna()
print(f"Trend Following Sharpe: {tf_ret.mean() / tf_ret.std():.4f}")
print(f"Days in market: {tf_signal.sum():.0f}/{len(tf_signal)} ({tf_signal.mean()*100:.1f}%)")

In [ ]:
def objective(trial, train_data):
    window = int(trial.suggest_categorical("window", [10, 15, 20, 25, 30, 35, 40, 50, 60]))
    signal, _, _ = donchian_breakout_signal(train_data, window=window)

    log_ret = np.log(train_data["Close"]).diff()
    strat_ret = signal.shift(1) * log_ret  # match backtest_strategy
    strat_ret = strat_ret.dropna()

    std = strat_ret.std()
    if std == 0 or np.isnan(std):
        return float("-inf")

    return strat_ret.mean() / std  # or annualize; ranking is unchanged up to sqrt(252)

In [ ]:
import optuna

# Walk-forward optimization for Trend Following strategy
study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: objective(trial, train_data), n_trials=100)

# Get best params
best_params = study.best_trial.params
print(f"Best params: {best_params}")

In [ ]:
tf_signal, donchian_high, donchian_low = donchian_breakout_signal(train_data, window=30)

# --- Visualize ---
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(train_data.index, train_data['Close'], color='gray', alpha=0.7, label='Close')
axes[0].plot(train_data.index, donchian_high, color='blue', lw=1, label='Donchian High (20)')
axes[0].plot(train_data.index, donchian_low, color='red', lw=1, label='Donchian Low (20)')
axes[0].fill_between(train_data.index, train_data['Close'].min(), train_data['Close'].max(),
                      where=tf_signal == 1, alpha=0.1, color='green', label='Long')
axes[0].set_title('Trend Following: Donchian Breakout (20-day)')
axes[0].set_ylabel('Price')
axes[0].legend(loc='upper left', fontsize=8)

axes[1].plot(train_data.index, donchian_high, color='blue', lw=1, label='Donchian High')
axes[1].plot(train_data.index, donchian_low, color='red', lw=1, label='Donchian Low')
axes[1].set_title('Donchian Channel Bounds')
axes[1].set_ylabel('Price')
axes[1].legend()

axes[2].plot(train_data.index, tf_signal, color='green', lw=0.8)
axes[2].set_title('Signal (1=Long, 0=Flat)')
axes[2].set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

# Quick perf
ret = np.log(train_data['Close']).diff().shift(-1)
tf_ret = (tf_signal * ret).dropna()
print(f"Trend Following Sharpe: {tf_ret.mean() / tf_ret.std():.4f}")
print(f"Days in market: {tf_signal.sum():.0f}/{len(tf_signal)} ({tf_signal.mean()*100:.1f}%)")

### Component 2: Mean Reversion — Bollinger Band Z-Score

In [ ]:
def mean_reversion_signal(df, bb_period=20, bb_std=2.0, entry_z=-1.0, exit_z=0.0):
    """
    Mean Reversion: Bollinger Band Z-Score.
    Long when Z < entry_z (oversold), exit when Z > exit_z (reverted).
    """
    close = df['Close']
    bb_upper, bb_mid, bb_lower = talib.BBANDS(close, timeperiod=bb_period,
                                               nbdevup=bb_std, nbdevdn=bb_std)
    bb_width = (bb_upper - bb_lower) / 2
    zscore = (close - bb_mid) / (bb_width + 1e-10)

    signal = pd.Series(0, index=df.index)
    in_pos = False

    for i in range(1, len(signal)):
        if not in_pos:
            if zscore.iloc[i] < entry_z:
                in_pos = True
                signal.iloc[i] = 1
        else:
            if zscore.iloc[i] > exit_z:
                in_pos = False
                signal.iloc[i] = 0
            else:
                signal.iloc[i] = 1

    return signal, zscore, bb_upper, bb_mid, bb_lower


mr_signal, zscore, bb_up, bb_mid, bb_low = mean_reversion_signal(train_data)

# --- Visualize ---
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(train_data.index, train_data['Close'], color='gray', alpha=0.7, label='Close')
axes[0].plot(train_data.index, bb_mid, color='blue', lw=1, label='BB Mid')
axes[0].fill_between(train_data.index, bb_low, bb_up, alpha=0.1, color='blue', label='BB ±2σ')
axes[0].fill_between(train_data.index, train_data['Close'].min(), train_data['Close'].max(),
                      where=mr_signal == 1, alpha=0.15, color='green', label='Long')
axes[0].set_title('Mean Reversion: Bollinger Band Z-Score')
axes[0].set_ylabel('Price')
axes[0].legend(loc='upper left', fontsize=8)

axes[1].plot(train_data.index, zscore, color='darkorange', lw=0.8)
axes[1].axhline(-1.0, color='green', ls='--', alpha=0.7, label='Entry Z=-1')
axes[1].axhline(0.0, color='red', ls='--', alpha=0.7, label='Exit Z=0')
axes[1].fill_between(train_data.index, zscore, -1.0, where=zscore < -1.0,
                      alpha=0.3, color='green', label='Oversold')
axes[1].set_title('Bollinger Band Z-Score')
axes[1].set_ylabel('Z-Score')
axes[1].set_ylim(-3, 3)
axes[1].legend(loc='upper right', fontsize=8)

axes[2].plot(train_data.index, mr_signal, color='blue', lw=0.8)
axes[2].set_title('Signal (1=Long, 0=Flat)')
axes[2].set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

mr_ret = (mr_signal * ret).dropna()
print(f"Mean Reversion Sharpe: {mr_ret.mean() / mr_ret.std():.4f}")
print(f"Days in market: {mr_signal.sum():.0f}/{len(mr_signal)} ({mr_signal.mean()*100:.1f}%)")

### Component 3: Regime Detector + Combined Signal

In [ ]:
def load_ihsg_close_aligned(index, buffer_days=400):
    """Download ^JKSE and align to the stock's trading calendar (forward-fill)."""
    start = (pd.Timestamp(index.min()) - pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d")
    end = (pd.Timestamp(index.max()) + pd.Timedelta(days=5)).strftime("%Y-%m-%d")
    ih = yf.download("^JKSE", start=start, end=end, auto_adjust=True, progress=False)["Close"]
    if isinstance(ih.columns, pd.MultiIndex):
        ih.columns = ih.columns.get_level_values(-1)
    ih = ih.squeeze().reindex(index).ffill().bfill()
    return ih


def volatility_position_scale(df, lookback=20, target_ann_vol=0.15):
    """Scale exposure down when recent realized vol exceeds target (vol targeting)."""
    r = np.log(df["Close"]).diff()
    realized_daily = r.rolling(lookback).std()
    target_daily = target_ann_vol / np.sqrt(252)
    scale = (target_daily / (realized_daily + 1e-8)).clip(upper=1.0)
    return scale.fillna(1.0)


def regime_switching_strategy_hurst(
    df, hurst_window=126, hurst_threshold=0.5,
    ema_short=12, ema_long=26, adx_period=14, adx_threshold=20,
    bb_period=20, bb_std=2.0, entry_z=-1.0, exit_z=0.0,
):
    """Legacy: rolling Hurst regime (slow). Returns integer 0/1 signal."""
    log_ret = np.log(df["Close"]).diff().dropna()
    regime = pd.Series(np.nan, index=df.index)
    for i in range(hurst_window, len(log_ret)):
        window_data = log_ret.values[i - hurst_window : i]
        try:
            h, _, _ = hurst_exponent(
                pd.Series(window_data), min_window=10, max_window=hurst_window // 4
            )
            regime.iloc[i + 1] = h
        except Exception:
            regime.iloc[i + 1] = np.nan

    is_trending = (regime > hurst_threshold).astype(float)
    tf_sig, _, _, _ = trend_following_signal(
        df, ema_short, ema_long, adx_period, adx_threshold
    )
    mr_sig, _, _, _, _ = mean_reversion_signal(
        df, bb_period, bb_std, entry_z, exit_z
    )
    combined = is_trending * tf_sig + (1.0 - is_trending) * mr_sig
    combined = combined.clip(0, 1)
    nan_mask = regime.isna()
    combined[nan_mask] = tf_sig[nan_mask]
    return combined.astype(int), regime, tf_sig, mr_sig, is_trending


def regime_switching_strategy(
    df,
    method="fast",
    hurst_window=126,
    hurst_threshold=0.5,
    ema_short=12,
    ema_long=26,
    adx_period=14,
    adx_threshold=20,
    bb_period=20,
    bb_std=2.0,
    entry_z=-1.0,
    exit_z=0.0,
    atr_fast=10,
    atr_slow=60,
    vol_ratio_threshold=1.0,
    sma_trend=200,
    use_ihsg_filter=True,
    ihsg_sma_period=50,
    ihsg_close=None,
    use_vol_target_sizing=True,
    target_ann_vol=0.15,
    vol_scale_lookback=20,
):
    """
    Regime-switching: trend-following vs mean-reversion.

    method='fast' (default): ATR short / ATR long ratio + price vs SMA(trend).
      - Trend regime: vol_ratio < threshold and Close > SMA (calm + structural uptrend).
      - Else: mean-reversion regime.

    method='hurst': slow rolling Hurst (legacy).

    use_ihsg_filter: only allow trend-following leg when ^JKSE > its SMA (market filter).
    use_vol_target_sizing: scale combined signal by inverse realized vol vs target.
    """
    tf_sig, _, _, _ = trend_following_signal(
        df, ema_short, ema_long, adx_period, adx_threshold
    )
    mr_sig, _, _, _, _ = mean_reversion_signal(
        df, bb_period, bb_std, entry_z, exit_z
    )

    if method == "hurst":
        return regime_switching_strategy_hurst(
            df,
            hurst_window,
            hurst_threshold,
            ema_short,
            ema_long,
            adx_period,
            adx_threshold,
            bb_period,
            bb_std,
            entry_z,
            exit_z,
        )

    high, low, close = df["High"], df["Low"], df["Close"]
    atr_f = talib.ATR(high, low, close, timeperiod=atr_fast)
    atr_s = talib.ATR(high, low, close, timeperiod=atr_slow)
    vol_ratio = atr_f / (atr_s + 1e-10)
    sma_long = talib.SMA(close, timeperiod=sma_trend)

    trend_regime_flag = (
        (vol_ratio < vol_ratio_threshold) & (close > sma_long)
    ).astype(float)
    regime = vol_ratio

    tf_eff = tf_sig.astype(float)
    if use_ihsg_filter:
        if ihsg_close is None:
            ihsg_close = load_ihsg_close_aligned(df.index)
        ihsg_ma = talib.SMA(ihsg_close.values, timeperiod=ihsg_sma_period)
        ihsg_ma = pd.Series(ihsg_ma, index=df.index)
        market_bull = (ihsg_close >= ihsg_ma).astype(float).fillna(1.0)
        tf_eff = tf_eff * market_bull

    combined = trend_regime_flag * tf_eff + (1.0 - trend_regime_flag) * mr_sig.astype(float)
    combined = combined.clip(0.0, 1.0)

    nan_regime = vol_ratio.isna() | sma_long.isna()
    combined = combined.where(~nan_regime, tf_eff)

    if use_vol_target_sizing:
        scale = volatility_position_scale(
            df, lookback=vol_scale_lookback, target_ann_vol=target_ann_vol
        )
        combined = (combined * scale).clip(0.0, 1.0)

    return combined, regime, tf_sig, mr_sig, trend_regime_flag


print("Computing regime-switching (fast ATR regime + IHSG filter + vol sizing)...")
rs_signal, regime, tf_sig, mr_sig, is_trending = regime_switching_strategy(
    train_data, method="fast"
)
rs_signal_legacy, _, _, _, _ = regime_switching_strategy(train_data, method="hurst")
print("Done!")

In [ ]:
# --- Visualize the regime-switching strategy ---
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# 1. Price with shaded regimes
axes[0].plot(train_data.index, train_data['Close'], color='gray', alpha=0.8, label='Close')
axes[0].fill_between(train_data.index, train_data['Close'].min(), train_data['Close'].max(),
                      where=is_trending == 1, alpha=0.08, color='red', label='Trending Regime')
axes[0].fill_between(train_data.index, train_data['Close'].min(), train_data['Close'].max(),
                      where=is_trending == 0, alpha=0.08, color='blue', label='Mean Reverting Regime')
axes[0].set_title('BBCA Price with Regime Overlay')
axes[0].set_ylabel('Price')
axes[0].legend(loc='upper left', fontsize=8)

# 2. Fast regime: ATR short / ATR long (volatility ratio)
axes[1].plot(train_data.index, regime, color='purple', lw=1.2, label='ATR fast / ATR slow')
axes[1].axhline(1.0, color='gray', ls='--', lw=1, label='Threshold = 1.0')
axes[1].fill_between(
    train_data.index,
    regime,
    1.0,
    where=(is_trending == 1) & regime.notna(),
    alpha=0.25,
    color='red',
    label='Trend regime (low vol ratio + price > SMA200)',
)
axes[1].fill_between(
    train_data.index,
    regime,
    1.0,
    where=(is_trending == 0) & regime.notna(),
    alpha=0.2,
    color='blue',
    label='Mean-reversion regime',
)
axes[1].set_title('Fast regime detector — volatility ratio')
axes[1].set_ylabel('Vol ratio')
axes[1].legend(loc='upper right', fontsize=8)

# 3. Individual signals
axes[2].plot(train_data.index, tf_sig * 0.9 + 0.05, color='red', alpha=0.5, lw=0.5, label='Trend Follow')
axes[2].plot(train_data.index, mr_sig * 0.4 - 0.45, color='blue', alpha=0.5, lw=0.5, label='Mean Reversion')
axes[2].axhline(0, color='gray', lw=0.5)
axes[2].set_title('Individual Strategy Signals')
axes[2].set_ylabel('Signal')
axes[2].legend()

# 4. Combined regime-switching signal (may be fractional after vol targeting)
axes[3].fill_between(train_data.index, 0, rs_signal, alpha=0.5, color='green', label='Position')
axes[3].set_title('Regime-switching position (incl. IHSG filter + vol sizing)')
axes[3].set_ylabel('Exposure')
axes[3].set_ylim(-0.05, 1.05)
axes[3].legend()

plt.tight_layout()
plt.show()

vr = regime.dropna()
print(f"Trend regime days:     {(is_trending == 1).mean()*100:.1f}%")
print(f"Mean-rev regime days:  {(is_trending == 0).mean()*100:.1f}%")
print(f"Latest vol ratio:      {vr.iloc[-1]:.4f}")

### Backtest: Performance Comparison

In [ ]:
def backtest_strategy(signal, df, transaction_cost=0.003, label="Strategy"):
    """
    Backtest long / flat / fractional exposure on log returns.

    Parameters
    ----------
    signal : pd.Series — position size in [0, 1] (0/1 binary or fractional vol targeting)
    df : pd.DataFrame with 'Close'
    transaction_cost : float — cost applied to |Δposition| each day (0.003 ≈ 0.3% per unit change)
    label : str
    """
    signal = signal.reindex(df.index).fillna(0.0)
    log_ret = np.log(df["Close"]).diff()
    strat_ret = signal.shift(1) * log_ret

    turnover = signal.diff().abs()
    costs = turnover * transaction_cost
    strat_ret_net = strat_ret - costs

    strat_ret_net = strat_ret_net.dropna()
    equity = strat_ret_net.cumsum()

    n_days = len(strat_ret_net)
    total_ret = equity.iloc[-1] if not equity.empty else 0
    ann_ret = total_ret / (n_days / 252) if n_days > 0 else 0
    ann_vol = strat_ret_net.std() * np.sqrt(252)
    sharpe = (
        (strat_ret_net.mean() / strat_ret_net.std()) * np.sqrt(252)
        if strat_ret_net.std() > 0
        else 0
    )

    cum_max = equity.cummax()
    drawdown = equity - cum_max
    max_dd = drawdown.min() if not drawdown.empty else 0

    winning = strat_ret_net[strat_ret_net > 0]
    losing = strat_ret_net[strat_ret_net < 0]
    win_rate = (
        len(winning) / (len(winning) + len(losing))
        if (len(winning) + len(losing)) > 0
        else 0
    )

    n_trades = int((turnover > 1e-6).sum())
    gross_profit = winning.sum()
    gross_loss = losing.abs().sum()
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf

    exposure = (signal > 1e-6).mean()

    return {
        "label": label,
        "equity": equity,
        "returns": strat_ret_net,
        "total_return": total_ret,
        "ann_return": ann_ret,
        "ann_volatility": ann_vol,
        "sharpe_ratio": sharpe,
        "max_drawdown": max_dd,
        "win_rate": win_rate,
        "n_trades": n_trades,
        "profit_factor": profit_factor,
        "days_in_market": exposure,
        "avg_exposure": float(signal.mean()),
    }


# --- Run backtests on training data ---
bh_signal = pd.Series(1.0, index=train_data.index)

results = {
    "Buy & Hold": backtest_strategy(bh_signal, train_data, 0, "Buy & Hold"),
    "Trend Following": backtest_strategy(tf_signal, train_data, 0.003, "Trend Following"),
    "Mean Reversion": backtest_strategy(mr_signal, train_data, 0.003, "Mean Reversion"),
    "Regime (legacy Hurst)": backtest_strategy(
        rs_signal_legacy, train_data, 0.003, "Regime legacy Hurst"
    ),
    "Regime (fast + IHSG + vol)": backtest_strategy(
        rs_signal, train_data, 0.003, "Regime fast"
    ),
}

# --- Performance table ---
metrics_df = pd.DataFrame({
    name: {
        "Total Return": f"{r['total_return']*100:.2f}%",
        "Ann. Return": f"{r['ann_return']*100:.2f}%",
        "Ann. Volatility": f"{r['ann_volatility']*100:.2f}%",
        "Sharpe Ratio": f"{r['sharpe_ratio']:.3f}",
        "Max Drawdown": f"{r['max_drawdown']*100:.2f}%",
        "Win Rate": f"{r['win_rate']*100:.1f}%",
        "Profit Factor": f"{r['profit_factor']:.3f}",
        "# Trades": r["n_trades"],
        "Days w/ Exposure": f"{r['days_in_market']*100:.1f}%",
        "Avg Exposure": f"{r['avg_exposure']*100:.1f}%",
    }
    for name, r in results.items()
})

print("=" * 80)
print("PERFORMANCE COMPARISON — Training Period")
print("=" * 80)
print(metrics_df.to_string())
print()

## Walk-forward optimization

Rolling **train** window (3 years of trading days), **test** window (6 months), **step** = test length. Each train segment: Hyperopt maximizes in-sample Sharpe on `regime_switching_strategy(..., use_ihsg_filter=False, use_vol_target_sizing=False)` for speed. Each test segment: apply best parameters with **IHSG filter + vol sizing** enabled (production settings). Stitched test returns form an out-of-sample equity curve, compared to **default-parameter** regime strategy on the same test segments.

### Permute Time Series
This function creates a randomized (permuted) version of an OHLC price series while preserving realistic market structure such as gaps and intrabar ranges.
It is a classic price-permutation / bootstrap technique often used in strategy validation and backtesting (e.g. similar to ideas in Marcos López de Prado).

In [ ]:
def get_permutation(
    df: pd.DataFrame,
    start_index: int = 0,
    seed: int | None = None
) -> pd.DataFrame:
    """
    Generate a permuted OHLC price series by shuffling log-return components
    while preserving realistic market structure (gaps + intrabar dynamics).

    Parameters
    ----------
    df : pd.DataFrame
        Original OHLC data with columns ['Open', 'High', 'Low', 'Close']
    start_index : int, default 0
        Index where permutation begins. Data before this index is kept intact.
    seed : int or None
        Random seed for reproducibility

    Returns
    -------
    pd.DataFrame
        Permuted OHLC DataFrame with the same index as the input
    """

    assert start_index >= 0, "start_index must be non-negative"
    np.random.seed(seed)

    # ------------------------------------------------------------------
    # Basic setup
    # ------------------------------------------------------------------
    time_index = df.index
    n_bars = len(df)

    # First bar that will be permuted
    perm_start = start_index + 1
    n_perm_bars = n_bars - perm_start

    # ------------------------------------------------------------------
    # Convert OHLC prices to log-space
    # ------------------------------------------------------------------
    log_ohlc = np.log(df[['Open', 'High', 'Low', 'Close']])

    # ------------------------------------------------------------------
    # Store the anchor bar (starting point for reconstruction)
    # ------------------------------------------------------------------
    start_bar_log = log_ohlc.iloc[start_index].to_numpy()

    # ------------------------------------------------------------------
    # Decompose price movements into log-return components
    # ------------------------------------------------------------------

    # Gap return: Open_t relative to Close_{t-1}
    rel_open = (log_ohlc['Open'] - log_ohlc['Close'].shift()).to_numpy()

    # Intrabar movements relative to Open_t
    rel_high = (log_ohlc['High'] - log_ohlc['Open']).to_numpy()
    rel_low = (log_ohlc['Low'] - log_ohlc['Open']).to_numpy()
    rel_close = (log_ohlc['Close'] - log_ohlc['Open']).to_numpy()

    # Keep only the portion that will be permuted
    rel_open = rel_open[perm_start:]
    rel_high = rel_high[perm_start:]
    rel_low = rel_low[perm_start:]
    rel_close = rel_close[perm_start:]

    # ------------------------------------------------------------------
    # Permute components independently
    # ------------------------------------------------------------------

    idx = np.arange(n_perm_bars)

    # Shuffle intrabar dynamics together (shape of candles)
    perm_intrabar = np.random.permutation(idx)
    rel_high = rel_high[perm_intrabar]
    rel_low = rel_low[perm_intrabar]
    rel_close = rel_close[perm_intrabar]

    # Shuffle gap returns independently
    perm_gap = np.random.permutation(idx)
    rel_open = rel_open[perm_gap]

    # ------------------------------------------------------------------
    # Reconstruct the permuted OHLC series (still in log-space)
    # ------------------------------------------------------------------
    perm_log_ohlc = np.zeros((n_bars, 4))

    # Copy real data before permutation start
    perm_log_ohlc[:start_index] = log_ohlc.iloc[:start_index].to_numpy()

    # Set anchor bar
    perm_log_ohlc[start_index] = start_bar_log

    # Sequentially rebuild prices
    for i in range(perm_start, n_bars):
        k = i - perm_start

        # Open_t = Close_{t-1} + gap
        perm_log_ohlc[i, 0] = perm_log_ohlc[i - 1, 3] + rel_open[k]

        # Intrabar prices relative to Open_t
        perm_log_ohlc[i, 1] = perm_log_ohlc[i, 0] + rel_high[k]   # High
        perm_log_ohlc[i, 2] = perm_log_ohlc[i, 0] + rel_low[k]    # Low
        perm_log_ohlc[i, 3] = perm_log_ohlc[i, 0] + rel_close[k]  # Close

    # ------------------------------------------------------------------
    # Convert back to price space
    # ------------------------------------------------------------------
    perm_ohlc = np.exp(perm_log_ohlc)

    return pd.DataFrame(
        perm_ohlc,
        index=time_index,
        columns=['Open', 'High', 'Low', 'Close']
    )

In [ ]:
df_perm = get_permutation(train_data)

real_r = np.log(train_data['Close']).diff()
perm_r = np.log(df_perm['Close']).diff()

print(f"Mean. REAL: {real_r.mean():14.6f} PERM: {perm_r.mean():14.6f}")
print(f"Stdd. REAL: {real_r.std():14.6f} PERM: {perm_r.std():14.6f}")
print(f"Skew. REAL: {real_r.skew():14.6f} PERM: {perm_r.skew():14.6f}")
print(f"Kurt. REAL: {real_r.kurt():14.6f} PERM: {perm_r.kurt():14.6f}")

np.log(train_data['Close']).diff().cumsum().plot(color='orange', label='Real')
np.log(df_perm['Close']).diff().cumsum().plot(color='purple', label='Permuted')

plt.ylabel("Cumulative Log Return")
plt.title("Real and Permuted BBCA Stock")
plt.legend()
plt.show()

Your permutation method preserves:

* What stays similar:
  * Return distribution (mean, variance, skew, kurtosis)
  * Volatility level
  * Candle shapes (intrabar high/low/close behavior)
  * Gap statistics (open vs previous close)
  * Price scale (log-space reconstruction)

* So when you plot cumulative log returns, both series:
  * Drift within a similar range
  * Exhibit similar drawdowns
  * Have comparable volatility clustering in appearance

* What is not preserved (and why it matters)
  * Autocorrelation
  * Trend persistence
  * Regime structure
  * Momentum / mean reversion
  * Any causal ordering

In [ ]:
from hyperopt import hp, fmin, tpe
from hyperopt.pyll import scope

def walk_forward_regime_backtest(
    ohlc_df,
    train_bars=252 * 3,
    test_bars=126,
    step_bars=None,
    transaction_cost=0.003,
    max_evals=30,
    random_state=42,
):
    """
    Walk-forward: optimize fast-regime parameters on each train window; apply on following test window.
    """
    if step_bars is None:
        step_bars = test_bars

    strat_space = {
        "ema_s": scope.int(hp.quniform("ema_s", 5, 16, 1)),
        "ema_l": scope.int(hp.quniform("ema_l", 18, 42, 1)),
        "adx_th": scope.int(hp.quniform("adx_th", 15, 28, 1)),
        "bb_p": scope.int(hp.quniform("bb_p", 12, 28, 1)),
        "ez": hp.uniform("ez", -2.0, -0.55),
        "vt": hp.uniform("vt", 0.82, 1.28),
    }

    wf_return_parts = []
    static_return_parts = []
    window_meta = []

    n = len(ohlc_df)
    pos = 0
    rng = np.random.default_rng(random_state)

    while pos + train_bars + test_bars <= n:
        train_slice = ohlc_df.iloc[pos : pos + train_bars]
        test_slice = ohlc_df.iloc[pos + train_bars : pos + train_bars + test_bars]

        def train_obj(p):
            es, el = int(p["ema_s"]), int(p["ema_l"])
            if es >= el - 1:
                return 1e6
            sig, _, _, _, _ = regime_switching_strategy(
                train_slice,
                method="fast",
                ema_short=es,
                ema_long=el,
                adx_threshold=float(p["adx_th"]),
                bb_period=int(p["bb_p"]),
                entry_z=float(p["ez"]),
                vol_ratio_threshold=float(p["vt"]),
                use_ihsg_filter=False,
                use_vol_target_sizing=False,
            )
            res = backtest_strategy(sig, train_slice, transaction_cost)
            return -float(res["sharpe_ratio"])

        best = fmin(
            train_obj,
            strat_space,
            algo=tpe.suggest,
            max_evals=max_evals,
            rstate=np.random.default_rng(int(rng.integers(0, 1_000_000))),
        )

        es, el = int(best["ema_s"]), int(best["ema_l"])
        if es >= el - 1:
            el = es + 5

        sig_wf, _, _, _, _ = regime_switching_strategy(
            test_slice,
            method="fast",
            ema_short=es,
            ema_long=el,
            adx_threshold=float(best["adx_th"]),
            bb_period=int(best["bb_p"]),
            entry_z=float(best["ez"]),
            vol_ratio_threshold=float(best["vt"]),
            use_ihsg_filter=True,
            use_vol_target_sizing=True,
        )
        res_wf = backtest_strategy(sig_wf, test_slice, transaction_cost)

        sig_st, _, _, _, _ = regime_switching_strategy(
            test_slice,
            method="fast",
            use_ihsg_filter=True,
            use_vol_target_sizing=True,
        )
        res_st = backtest_strategy(sig_st, test_slice, transaction_cost)

        sig_tr_in, _, _, _, _ = regime_switching_strategy(
            train_slice,
            method="fast",
            ema_short=es,
            ema_long=el,
            adx_threshold=float(best["adx_th"]),
            bb_period=int(best["bb_p"]),
            entry_z=float(best["ez"]),
            vol_ratio_threshold=float(best["vt"]),
            use_ihsg_filter=False,
            use_vol_target_sizing=False,
        )
        train_sh_in = backtest_strategy(sig_tr_in, train_slice, transaction_cost)[
            "sharpe_ratio"
        ]

        wf_return_parts.append(res_wf["returns"])
        static_return_parts.append(res_st["returns"])
        window_meta.append(
            {
                "test_start": test_slice.index[0],
                "test_end": test_slice.index[-1],
                "train_sharpe_opt": train_sh_in,
                "wf_test_sharpe": res_wf["sharpe_ratio"],
                "static_test_sharpe": res_st["sharpe_ratio"],
                "params": dict(best),
            }
        )

        pos += step_bars

    wf_rets = pd.concat(wf_return_parts) if wf_return_parts else pd.Series(dtype=float)
    st_rets = pd.concat(static_return_parts) if static_return_parts else pd.Series(dtype=float)

    wf_equity = wf_rets.cumsum()
    st_equity = st_rets.cumsum()

    summary = pd.DataFrame(window_meta)
    return {
        "wf_returns": wf_rets,
        "wf_equity": wf_equity,
        "static_oos_returns": st_rets,
        "static_oos_equity": st_equity,
        "windows": summary,
    }


print("Running walk-forward (Hyperopt per window; may take several minutes)...")
wf_out = walk_forward_regime_backtest(
    data,
    train_bars=252 * 3,
    test_bars=126,
    step_bars=126,
    max_evals=25,
)
print(f"Completed {len(wf_out['windows'])} walk-forward windows.")
print(
    wf_out["windows"][
        ["test_start", "test_end", "train_sharpe_opt", "wf_test_sharpe", "static_test_sharpe"]
    ]
    .round(3)
    .to_string()
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(wf_out["wf_equity"].index, wf_out["wf_equity"], label="Walk-forward OOS (re-optimized)", lw=2)
ax.plot(
    wf_out["static_oos_equity"].index,
    wf_out["static_oos_equity"],
    label="Default params (same OOS segments)",
    lw=1.5,
    alpha=0.75,
)
ax.set_title("Stitched out-of-sample cumulative log returns — walk-forward vs static defaults")
ax.set_ylabel("Cumulative log return")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

wf_sh = (
    wf_out["wf_returns"].mean() / wf_out["wf_returns"].std() * np.sqrt(252)
    if len(wf_out["wf_returns"]) > 10 and wf_out["wf_returns"].std() > 0
    else np.nan
)
st_sh = (
    wf_out["static_oos_returns"].mean()
    / wf_out["static_oos_returns"].std()
    * np.sqrt(252)
    if len(wf_out["static_oos_returns"]) > 10 and wf_out["static_oos_returns"].std() > 0
    else np.nan
)
print(f"Aggregated OOS Sharpe (WF):    {wf_sh:.3f}")
print(f"Aggregated OOS Sharpe (static): {st_sh:.3f}")

### Permutation Validation: Is the Edge Real?

In [ ]:
def permutation_test(strategy_func, df, n_perms=100, transaction_cost=0.003):
    """
    Run strategy on real data, then on N permuted series.
    Returns real Sharpe and distribution of permuted Sharpes.
    """
    # Real performance
    real_signal = strategy_func(df)
    real_result = backtest_strategy(real_signal, df, transaction_cost, "Real")
    real_sharpe = real_result['sharpe_ratio']

    # Permuted performances
    perm_sharpes = []
    for i in range(n_perms):
        perm_df = get_permutation(df, seed=i)
        try:
            perm_signal = strategy_func(perm_df)
            perm_result = backtest_strategy(perm_signal, perm_df, transaction_cost, f"Perm {i}")
            perm_sharpes.append(perm_result['sharpe_ratio'])
        except:
            perm_sharpes.append(0.0)

    perm_sharpes = np.array(perm_sharpes)
    p_value = np.mean(perm_sharpes >= real_sharpe)

    return real_sharpe, perm_sharpes, p_value


# Wrapper functions that return just the signal
def rs_wrapper(df):
    sig, _, _, _, _ = regime_switching_strategy(
        df,
        method="fast",
        use_ihsg_filter=False,
        use_vol_target_sizing=False,
    )
    return sig

def tf_wrapper(df):
    sig, _, _, _ = trend_following_signal(df)
    return sig

def mr_wrapper(df):
    sig, _, _, _, _ = mean_reversion_signal(df)
    return sig


print("Running permutation tests (this will take several minutes)...")
print()

N_PERMS = 50  # Reduce for speed; increase to 200+ for publication

print("1/3 Testing Trend Following...")
tf_real, tf_perms, tf_pval = permutation_test(tf_wrapper, train_data, N_PERMS)

print("2/3 Testing Mean Reversion...")
mr_real, mr_perms, mr_pval = permutation_test(mr_wrapper, train_data, N_PERMS)

print("3/3 Testing Regime (fast, simplified for permutation)...")
rs_real, rs_perms, rs_pval = permutation_test(rs_wrapper, train_data, N_PERMS)

print("Done!")

In [ ]:
# --- Permutation Test Results ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, real, perms, pval) in zip(axes, [
    ('Trend Following', tf_real, tf_perms, tf_pval),
    ('Mean Reversion', mr_real, mr_perms, mr_pval),
    ('Regime (fast, no IHSG/vol)', rs_real, rs_perms, rs_pval)
]):
    ax.hist(perms, bins=20, alpha=0.6, color='gray', edgecolor='black', label='Permuted')
    ax.axvline(real, color='red', lw=2, ls='--', label=f'Real = {real:.3f}')
    ax.set_title(f'{name}\np-value = {pval:.3f}')
    ax.set_xlabel('Annualized Sharpe Ratio')
    ax.legend(fontsize=8)

plt.suptitle('Permutation Test — Is the Strategy Edge Real?', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("=" * 60)
print("PERMUTATION TEST RESULTS")
print("=" * 60)
for name, real, pval in [('Trend Following', tf_real, tf_pval),
                          ('Mean Reversion', mr_real, mr_pval),
                          ('Regime (fast)', rs_real, rs_pval)]:
    status = "REAL EDGE" if pval < 0.05 else "NOT SIGNIFICANT"
    symbol = "pass" if pval < 0.05 else "fail"
    print(f"  {name:20s}  Sharpe={real:+.3f}  p={pval:.3f}  [{status}]")

### Out-of-Sample Test

In [ ]:
# --- Run all strategies on the TEST set (out-of-sample) ---
print("Running strategies on OUT-OF-SAMPLE test data...")
print()

# Generate signals on test data
tf_signal_oos, _, _, _ = trend_following_signal(test_data)
mr_signal_oos, _, _, _, _ = mean_reversion_signal(test_data)

print("Computing regime-switching on test data...")
rs_signal_oos, regime_oos, _, _, is_trending_oos = regime_switching_strategy(
    test_data, method="fast"
)
rs_legacy_oos, _, _, _, _ = regime_switching_strategy(test_data, method="hurst")

bh_signal_oos = pd.Series(1.0, index=test_data.index)

oos_results = {
    "Buy & Hold": backtest_strategy(bh_signal_oos, test_data, 0, "Buy & Hold"),
    "Trend Following": backtest_strategy(tf_signal_oos, test_data, 0.003, "Trend Following"),
    "Mean Reversion": backtest_strategy(mr_signal_oos, test_data, 0.003, "Mean Reversion"),
    "Regime (legacy Hurst)": backtest_strategy(
        rs_legacy_oos, test_data, 0.003, "Regime legacy OOS"
    ),
    "Regime (fast + IHSG + vol)": backtest_strategy(
        rs_signal_oos, test_data, 0.003, "Regime fast OOS"
    ),
}

# Performance table
oos_metrics = pd.DataFrame({
    name: {
        'Total Return': f"{r['total_return']*100:.2f}%",
        'Ann. Return': f"{r['ann_return']*100:.2f}%",
        'Sharpe Ratio': f"{r['sharpe_ratio']:.3f}",
        'Max Drawdown': f"{r['max_drawdown']*100:.2f}%",
        'Profit Factor': f"{r['profit_factor']:.3f}",
        '# Trades': r['n_trades'],
    }
    for name, r in oos_results.items()
})

print("=" * 80)
print("OUT-OF-SAMPLE PERFORMANCE — Test Period")
print("=" * 80)
print(oos_metrics.to_string())
print()

fig, ax = plt.subplots(figsize=(14, 6))

oos_palette = {
    "Buy & Hold": "gray",
    "Trend Following": "red",
    "Mean Reversion": "blue",
    "Regime (legacy Hurst)": "orange",
    "Regime (fast + IHSG + vol)": "green",
}
hl = "Regime (fast + IHSG + vol)"
for i, (name, r) in enumerate(oos_results.items()):
    col = oos_palette.get(name, plt.cm.tab10(i % 10))
    ax.plot(
        r["equity"].index,
        r["equity"],
        label=name,
        color=col,
        lw=2.2 if name == hl else 1,
        alpha=1.0 if name == hl else 0.55,
    )

ax.set_title("Out-of-Sample Equity Curves — Test Period")
ax.set_ylabel('Cumulative Log Return')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Final verdict
print()
print("=" * 60)
sharpes = {n: r['sharpe_ratio'] for n, r in oos_results.items() if n != 'Buy & Hold'}
best = max(sharpes, key=sharpes.get)
print(f"Best OOS strategy: {best} (Sharpe = {sharpes[best]:.3f})")
print("=" * 60)